In [7]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [2]:
mpl.style.use('default')
mpl.rcParams['axes.linewidth'] = 7  # set the value globally
mpl.rcParams['xtick.major.size'] = 24
mpl.rcParams['xtick.major.width'] = 7
mpl.rcParams['xtick.minor.size'] = 16
mpl.rcParams['xtick.minor.width'] = 7
mpl.rcParams['ytick.major.size'] = 24
mpl.rcParams['ytick.major.width'] = 7
mpl.rcParams['ytick.labelsize'] = 75
mpl.rcParams['xtick.labelsize'] = 75
mpl.rcParams['ytick.minor.size'] = 16
mpl.rcParams['ytick.minor.width'] = 7
mpl.rcParams['font.size'] = 55
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['svg.fonttype'] = 'none'

In [3]:
inf_df = pd.read_pickle(r"nontreated_vs_infected.pkl")

In [8]:
N=12
from matplotlib.colors import to_hex

mpl.rcParams['figure.dpi'] = 500
save_plot =True

line_colors = [
    to_hex(c) for c in plt.cm.YlOrRd(np.linspace(0.8, 0.25, N))
]
base_position = inf_df['base_position'].unique()
line_dict_color = {}
for base, color_o in zip(base_position, line_colors):
    line_dict_color[base] = color_o
    
unique_parameters = inf_df.experimentparameter.unique()
color_palette = ['k', 'purple']
colors = {}

for i, param in enumerate(unique_parameters):
    colors[param] = color_palette[i]

construct = len(inf_df.construct.unique())
parameter = len(inf_df.experimentparameter.unique())


ylim= [ymin, ymax] = -0.12, 0.13
 

uni_construct = inf_df.construct.unique()

#Added
func_1 = np.vectorize(lambda x : ((x-1) *20 + 1))
func_2 = np.vectorize(lambda x : ((x) * 20 + 44))

construct_array = np.array([int(x) for x in uni_construct])

pos_1 = func_1(construct_array)
pos_2 = func_2(construct_array)

xlabel_list = []
for i in range(12):
    xlabel_list.append('T'+str(i+1))
###

#indexer length

fig, ax = plt.subplots(1, 1, figsize=(30, 15))

y=0
con_holder = None
for x in inf_df['plot_position']:
    current_well_n = inf_df.loc[inf_df['plot_position']==x, 'well_count'].iloc[0]
    ax.text(x, ymin+0.02, f'n={current_well_n}', ha='center', fontsize=24)
    
    len_param = inf_df.experimentparameter.nunique()
    current_construct = inf_df.loc[inf_df['plot_position']==x, 'construct'].iloc[0]
    if con_holder is None:
        con_holder = current_construct
        comparison_positions = x #added
        y+=1
        continue
    
    if con_holder != current_construct:
        y=1
        con_holder = current_construct
        comparison_positions = x #added
        continue
    
    if y == len_param:
        y=1
        continue
        
        
    y_pos = (ymax-0.01) - y*0.0078
    x_coords = (comparison_positions, x)    #added
    
    
    star_val = inf_df.loc[inf_df['plot_position']==x, 'star_value']
    #ax[row].text(x, y_pos, s=star_val.iloc[0], fontsize=8, fontweight='bold', ha='center')

    ax.plot(x_coords, (y_pos, y_pos), linewidth=2, c='k') #added
    
    center_x = sum(x_coords)/len(x_coords) #added
    ax.text(center_x, y_pos+0.003, s=star_val.iloc[0], fontsize=28, fontweight='bold', ha='center') #change
    
    y+=1
ax.set_ylim(ylim)
tmp_explode = inf_df.explode('delta_Efret_median_well')

x_positions = tmp_explode['plot_position'] + np.random.uniform(-0.05, 0.05, size=len(tmp_explode['plot_position']))
ax.scatter(x=x_positions , y=tmp_explode['delta_Efret_median_well'], s=300, c='white', edgecolor='navy', linewidth=7, zorder=2, alpha=0.4)

inf_df['delta_Efret_median'] = inf_df['delta_Efret_median_well'].apply(np.median)
inf_df = inf_df.reset_index(drop=True)
inf_df['std_median_per_timepoint'] = inf_df.explode(['delta_Efret_median_well']).groupby(['construct', 'experimentparameter'], as_index=False)['delta_Efret_median_well'].std().dropna()['delta_Efret_median_well'].reset_index(drop=True)

ax.scatter(x= inf_df['plot_position'], y=inf_df['delta_Efret_median'], s=900, c='gray', zorder=3, edgecolor='k', linewidth=3)
ax.errorbar(x= inf_df['plot_position'], y=inf_df['delta_Efret_median'], yerr= inf_df['std_median_per_timepoint'], linestyle='none', color='k',elinewidth=8, zorder=3)
tmp_explode = inf_df.explode('delta_Efret_list_well')

iterator = 0 
 
y_plot_position = []
column_iter = 'plot_position'
for x in inf_df[column_iter]:
    list_explode = tmp_explode.loc[tmp_explode[column_iter]== x]
    current_parameter = tmp_explode.loc[tmp_explode[column_iter]== x, 'experimentparameter'].iloc[0]
    base_position = tmp_explode.loc[tmp_explode[column_iter]== x, 'base_position'].iloc[0]
    base_y_position = tmp_explode.loc[tmp_explode[column_iter]== x, 'delta_Efret_median'].iloc[0]
    
    len_dataframe = len(list_explode)
    
    len_para = tmp_explode.experimentparameter.nunique()
    if iterator == len_para:
        iterator = 0
    
    match current_parameter:
        case 'Growth':
            side='low'
            iterator = 0
            y_plot_position = [base_y_position]
            x_plot_position = [x]
        case _:
            side='high'
            y_plot_position.append(base_y_position)
            x_plot_position.append(x)
            current_line_color = line_dict_color[base_position]
            ax.plot(x_plot_position, y_plot_position, c=current_line_color, linewidth=5)
    current_color = colors[current_parameter]
    
    for sub in range(len_dataframe):
        current_dataframe = list_explode.iloc[sub]
        
        final_explode = current_dataframe['delta_Efret_list_well']
        
        violins =ax.violinplot(final_explode,[base_position], widths=0.9, showextrema=False, showmedians=False, showmeans=False, side=side)

        
        for pc in violins['bodies']:
            pc.set_facecolor('None')
            pc.set_edgecolor(current_color)
            pc.set_aplha=0.3
            pc.set_linewidth(6)

    iterator += 1

ax.set_xticks(np.arange(1, construct+1, 1))

ax.set_xticklabels(xlabel_list, fontsize='large')




ax.set_ylabel('Delta Efret')
ax.set_xlabel('Linker Sequence')

patch = []
labels = ['Nontreated', 'High Infection Marker']
for color, label in zip(color_palette, labels):
    
    tmp_patch = mpatches.Patch(color=color, label=label)
    patch.append(tmp_patch)
plt.legend(title='mCherry Intensity', handles=patch, loc='upper right', bbox_to_anchor=(1.45, 1))

plt.subplots_adjust(hspace=1)
fig.suptitle('Infection: AdV5-'+'\u0394'+'E3 (Post Infection)', x=0.5, y=0.95)

#plt.savefig(r'infection_violin_figure_NontreatedvsHigh.svg', format='svg', bbox_inches='tight')
plt.show()

C:\Users\Owner\AppData\Local\Temp\ipykernel_20260\2380495808.py:95: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  inf_df['std_median_per_timepoint'] = inf_df.explode(['delta_Efret_median_well']).groupby(['construct', 'experimentparameter'], as_index=False)['delta_Efret_median_well'].std().dropna()['delta_Efret_median_well'].reset_index(drop=True)


In [9]:
hist_df = pd.read_pickle("histogram_df.pkl")

In [11]:
fig, ax  = plt.subplots(2, sharex=True, figsize=(20, 24))

bins = np.logspace(np.log10(100), np.log10(65355), 100)

colors = ['limegreen', 'turquoise', 'deepskyblue']
group_values = ['experimentparameter', 'timepoint']

len_unique_parms = list(hist_df['experimentparameter'].unique())
len_unique_time = list(hist_df['timepoint'].unique())

df_generator = hist_df.groupby(group_values)

for index, container in enumerate(df_generator):
    #get current parm to select current row
    iter_parameter = container[0][group_values.index('experimentparameter')]
    para_index = len_unique_parms.index(iter_parameter)
    #get current time to select current color
    time = container[0][group_values.index('timepoint')]
    time_index = len_unique_time.index(time)
    color = colors[time_index]
    
    tmp_df = container[1]
    
    ax[para_index].hist(tmp_df['mCherry_intensity_mean'],  bins=bins, histtype='step', color=color, linewidth=4)
    ax[para_index].axvline(300, linestyle='--', c='k', linewidth=3)

    ax[para_index].set_ylim(3, 20000)
    ax[para_index].set_xlim(100, 65355)
    ax[para_index].set_ylabel('Cell Count', fontsize=75)
    ax[para_index].tick_params(axis='both', which='major', labelsize=58)
    yticklabels = ax[para_index].get_yticks()
    xticklabels = ax[para_index].get_xticks()
    
    #ax[para_index].set_xticklabels(xticklabels, fontsize=58)
    #ax[para_index].set_yticklabels(yticklabels, fontsize=58)

    ax[para_index].set_xscale('log')
    ax[para_index].set_yscale('log')
    
    ax2 = ax[para_index].twinx()
    ax2.set_yticks([])
    ax2.set_ylabel(f'{iter_parameter}')
    

ax[para_index].set_xlabel('Mean mCherry Intensity (AU)', fontsize=75)

color_palette = ['k', 'dodgerblue', 'firebrick']

ax[0].axvspan(100, 300, color=color_palette[0], alpha=0.1)
ax[1].axvspan(100, 300, color=color_palette[1], alpha=0.1)
ax[1].axvspan(2000, 65535, color='purple', alpha=0.1)


patch = []
labels = ['12 Hours', '24 hours', '48 Hours']
for color, label in zip(colors, labels):
    tmp_patch = mpatches.Patch(color=color, label=label)
    patch.append(tmp_patch)
plt.legend(title='Time (HPI)', handles=patch, loc='upper right', bbox_to_anchor=(0.9,2.1))
#plt.savefig(r'Infection_histograms_time_overlays.svg', format='svg', bbox_inches='tight')
plt.show()